CS4001/4042 Assignment 1
---
Part B, Q1 (15 marks)
---

Real world datasets often have a mix of numeric and categorical features – this dataset is one example. To build models on such data, categorical features have to be encoded or embedded.

PyTorch Tabular is a library that makes it very convenient to build neural networks for tabular data. It is built on top of PyTorch Lightning, which abstracts away boilerplate model training code and makes it easy to integrate other tools, e.g. TensorBoard for experiment tracking.

For questions B1 and B2, the following features should be used:   
- **Numeric / Continuous** features: dist_to_nearest_stn, dist_to_dhoby, degree_centrality, eigenvector_centrality, remaining_lease_years, floor_area_sqm
- **Categorical** features: month, town, flat_model_type, storey_range



---



In [1]:
!pip install pytorch_tabular[extra]

In [2]:
SEED = 42

import os

import random
random.seed(SEED)

import numpy as np
np.random.seed(SEED)

import pandas as pd

import torch
import torch.nn as nn

from pytorch_tabular import TabularModel
from pytorch_tabular.models import CategoryEmbeddingModelConfig
from pytorch_tabular.config import (
    DataConfig,
    OptimizerConfig,
    TrainerConfig,
)

> Divide the dataset (‘hdb_price_prediction.csv’) into train and test sets by using entries from year 2020 and before as training data, and year 2021 as test data (validation set is not required).
**Do not** use data from year 2022 and year 2023.



In [4]:
df = pd.read_csv('hdb_price_prediction.csv')

# TODO: Enter your code here

# Split the dataset
train_df = df[df["year"] <= 2020]  # Training data (2020 and before)
test_df = df[df["year"] == 2021]   # Test data (only 2021)

# Check the number of entries in each set
train_size = train_df.shape[0]
test_size = test_df.shape[0]

print(f"Training set size: {train_size}")
print(f"Test set size: {test_size}")

Training set size: 87370
Test set size: 29057


> Refer to the documentation of **PyTorch Tabular** and perform the following tasks: https://pytorch-tabular.readthedocs.io/en/latest/#usage
- Use **[DataConfig](https://pytorch-tabular.readthedocs.io/en/latest/data/)** to define the target variable, as well as the names of the continuous and categorical variables.
- Use **[TrainerConfig](https://pytorch-tabular.readthedocs.io/en/latest/training/)** to automatically tune the learning rate. Set batch_size to be 1024 and set max_epoch as 50.
- Use **[CategoryEmbeddingModelConfig](https://pytorch-tabular.readthedocs.io/en/latest/models/#category-embedding-model)** to create a feedforward neural network with 1 hidden layer containing 50 neurons.
- Use **[OptimizerConfig](https://pytorch-tabular.readthedocs.io/en/latest/optimizer/)** to choose Adam optimiser. There is no need to set the learning rate (since it will be tuned automatically) nor scheduler.
- Use **[TabularModel](https://pytorch-tabular.readthedocs.io/en/latest/tabular_model/)** to initialise the model and put all the configs together.

In [5]:
# TODO: Enter your code here

# Define the target variable and features
data_config = DataConfig(
    target=["resale_price"],  # Target variable
    continuous_cols=[
        "dist_to_nearest_stn", "dist_to_dhoby", "degree_centrality",
        "eigenvector_centrality", "remaining_lease_years", "floor_area_sqm"
    ],  # Continuous variables
    categorical_cols=["town", "full_address", "nearest_stn", "flat_model_type", "storey_range"],  # Categorical variables
)

# Trainer configuration with learning rate tuning
trainer_config = TrainerConfig(
    auto_lr_find=True,  # Automatically tune learning rate
    batch_size=1024,
    max_epochs=50,
)

# Neural network model configuration
model_config = CategoryEmbeddingModelConfig(
    task="regression",
    # layers=[50],  # One hidden layer with 50 neurons
    layers="50",  # Correct: String format
    activation="ReLU",
)

# Optimizer configuration
optimizer_config = OptimizerConfig(
    optimizer="Adam",  # Adam optimizer
)

# Initialize the TabularModel with all configurations
tabular_model = TabularModel(
    data_config=data_config,
    model_config=model_config,
    optimizer_config=optimizer_config,
    trainer_config=trainer_config,
)

print(tabular_model)

INFO:pytorch_tabular.tabular_model:Experiment Tracking is turned off


TabularModel(model=CategoryEmbeddingModel(Not Initialized))


> Report the test RMSE error and the test R2 value that you obtained.



In [6]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Fit the model
tabular_model.fit(train=train_df)

# Make predictions
predictions = tabular_model.predict(test_df)

# Debug: Print column names
print(predictions.columns)

# Extract actual and predicted values
y_true = test_df["resale_price"].values
y_pred = predictions.iloc[:, 0].values  # Adjust based on actual output

# Compute RMSE
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

# Compute R² score
r2 = r2_score(y_true, y_pred)

print(f"Test RMSE: {rmse:.4f}")
print(f"Test R²: {r2:.4f}")


INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_tabular.tabular_model:Preparing the DataLoaders
INFO:pytorch_tabular.tabular_datamodule:Setting up the datamodule for regression task
/usr/local/lib/python3.11/dist-packages/pytorch_tabular/categorical_encoders.py:71: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_encoded[col].fillna(self._imputed, inplace=True)
/usr/local/lib/python3.11/dist-packages/pytorch_tabular/categorical_encoders.py:71: FutureWarning: A value is trying to be set on a copy of a DataFrame 

Finding best initial lr:   0%|          | 0/100 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=100` reached.
INFO:pytorch_lightning.tuner.lr_finder:Learning rate set to 0.47863009232263803
INFO:pytorch_lightning.utilities.rank_zero:Restoring states from the checkpoint path at /content/.lr_find_e62b158b-91f7-4160-8343-5cb817b2b673.ckpt
INFO:pytorch_lightning.utilities.rank_zero:Restored all states from the checkpoint at /content/.lr_find_e62b158b-91f7-4160-8343-5cb817b2b673.ckpt
INFO:pytorch_tabular.tabular_model:Suggested LR: 0.47863009232263803. For plot and detailed analysis, use `find_learning_rate` method.
INFO:pytorch_tabular.tabular_model:Training Started
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type                      ┃ Params ┃ Mode  ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ 0 │ _backbone        │ CategoryEmbeddingBackbone │  7.3 K │ train │
│ 1 │ _embedding_layer │ Embedding1dLayer          │  448 K │ train │
│ 2 │ head             │ LinearHead                │     51 │ train │
│ 3 │ loss             │ MSELoss                   │      0 │ train │
└───┴──────────────────┴───────────────────────────┴────────┴───────┘

Trainable params: 455 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 455 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 17                                                                                          
Modules in eval mode: 0

Output()

INFO:pytorch_tabular.tabular_model:Training the model completed
INFO:pytorch_tabular.tabular_model:Loading the best model
/usr/local/lib/python3.11/dist-packages/pytorch_tabular/utils/python_utils.py:85: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please op

Index(['resale_price_prediction'], dtype='object')
Test RMSE: 73491.3736
Test R²: 0.7958


> Print out the corresponding rows in the dataframe for the top 25 test samples with the largest errors.



In [7]:
# TODO: Enter your code here
import pandas as pd

# Compute absolute errors
test_df["absolute_error"] = np.abs(y_true - y_pred)

# Sort by absolute error (largest first)
top_25_errors = test_df.sort_values(by="absolute_error", ascending=False).head(25)

# Print the top 25 rows with the largest errors
print(top_25_errors)


        month  year             town               full_address  \
92226       9  2021      BUKIT MERAH         96A HENDERSON ROAD   
90608      12  2021           BISHAN      273B BISHAN STREET 24   
92442      11  2021      BUKIT MERAH         127D KIM TIAN ROAD   
92443      11  2021      BUKIT MERAH         96A HENDERSON ROAD   
96910      12  2021          GEYLANG           332 UBI AVENUE 1   
100836      6  2021  KALLANG/WHAMPOA           39 JALAN BAHAGIA   
106132     11  2021       QUEENSTOWN      50 COMMONWEALTH DRIVE   
90432       8  2021           BISHAN      275A BISHAN STREET 24   
105702      6  2021       QUEENSTOWN        150 MEI LING STREET   
90523      10  2021           BISHAN      273B BISHAN STREET 24   
90431       8  2021           BISHAN      273A BISHAN STREET 24   
92340      10  2021      BUKIT MERAH           56 HAVELOCK ROAD   
90483       9  2021           BISHAN      273A BISHAN STREET 24   
114389     10  2021        WOODLANDS    805 WOODLANDS STREET 8

<ipython-input-7-292c2f06e0fe>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df["absolute_error"] = np.abs(y_true - y_pred)


Part B, Q2 (10 marks)
---
In Question B1, we used the Category Embedding model. This creates a feedforward neural network in which the categorical features get learnable embeddings. In this question, we will make use of a library called Pytorch-WideDeep. This library makes it easy to work with multimodal deep-learning problems combining images, text, and tables. We will just be utilizing the deeptabular component of this library through the TabMlp network.

In [8]:
!pip install pytorch-widedeep

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.0/22.0 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.4/38.4 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 71.2 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.14.1
    Uninstalling scipy-1.14.1:
      Successfully uninstalled scipy-1.14.1


In [7]:
from pytorch_widedeep.preprocessing import TabPreprocessor
from pytorch_widedeep.models import TabMlp, WideDeep
from pytorch_widedeep import Trainer
from pytorch_widedeep.metrics import R2Score

>Divide the dataset (‘hdb_price_prediction.csv’) into train and test sets by using entries from the year 2020 and before as training data, and entries from 2021 and after as the test data（validation set is not required here).

In [8]:
# Install the required library
!pip install pytorch-widedeep

import pandas as pd
import numpy as np
import torch
from pytorch_widedeep.preprocessing import TabPreprocessor
from pytorch_widedeep.models import TabMlp, WideDeep
from pytorch_widedeep import Trainer
from pytorch_widedeep.metrics import R2Score
from sklearn.metrics import mean_squared_error

# 1️⃣ Load the dataset
df = pd.read_csv('hdb_price_prediction.csv')

# 2️⃣ Ensure 'year' column is integer
df['year'] = pd.to_numeric(df['year'], errors='coerce')  # Convert year column if needed

# 3️⃣ Split dataset
train_df = df[df['year'] <= 2020]  # Training: 2020 and before
test_df = df[df['year'] >= 2021]   # Testing: 2021 and after

print(test_df)

        month  year        town              full_address   nearest_stn  \
87370       1  2021  ANG MO KIO   170 ANG MO KIO AVENUE 4  Yio Chu Kang   
87371       1  2021  ANG MO KIO   170 ANG MO KIO AVENUE 4  Yio Chu Kang   
87372       1  2021  ANG MO KIO   331 ANG MO KIO AVENUE 1    Ang Mo Kio   
87373       1  2021  ANG MO KIO  534 ANG MO KIO AVENUE 10    Ang Mo Kio   
87374       1  2021  ANG MO KIO  561 ANG MO KIO AVENUE 10    Ang Mo Kio   
...       ...   ...         ...                       ...           ...   
159548      8  2023      YISHUN      344 YISHUN AVENUE 11        Yishun   
159549      8  2023      YISHUN        325 YISHUN CENTRAL        Yishun   
159550      8  2023      YISHUN        325 YISHUN CENTRAL        Yishun   
159551      8  2023      YISHUN      387 YISHUN RING ROAD        Yishun   
159552      8  2023      YISHUN      277 YISHUN STREET 22        Yishun   

        dist_to_nearest_stn  dist_to_dhoby  degree_centrality  \
87370              1.276775       

>Refer to the documentation of Pytorch-WideDeep and perform the following tasks:
https://pytorch-widedeep.readthedocs.io/en/latest/index.html
* Use [**TabPreprocessor**](https://pytorch-widedeep.readthedocs.io/en/latest/examples/01_preprocessors_and_utils.html#2-tabpreprocessor) to create the deeptabular component using the continuous
features and the categorical features. Use this component to transform the training dataset.
* Create the [**TabMlp**](https://pytorch-widedeep.readthedocs.io/en/latest/pytorch-widedeep/model_components.html#pytorch_widedeep.models.tabular.mlp.tab_mlp.TabMlp) model with 2 hidden layers in the MLP, with 200 and 100 neurons respectively.
* Create a [**Trainer**](https://pytorch-widedeep.readthedocs.io/en/latest/pytorch-widedeep/trainer.html#pytorch_widedeep.training.Trainer) for the training of the created TabMlp model with the root mean squared error (RMSE) cost function. Train the model for 60 epochs using this trainer, keeping a batch size of 64. (Note: set the *num_workers* parameter to 0.)

In [17]:

from pytorch_widedeep.preprocessing import TabPreprocessor
from pytorch_widedeep.models import TabMlp, WideDeep
from pytorch_widedeep.metrics import R2Score
from pytorch_widedeep.training import Trainer  # ✅ Correct


# Define categorical and continuous columns
cat_embed_cols = ["town", "full_address", "nearest_stn", "flat_model_type", "storey_range"]
continuous_cols = ["dist_to_nearest_stn", "dist_to_dhoby", "degree_centrality", "eigenvector_centrality", "remaining_lease_years", "floor_area_sqm"]

# Use TabPreprocessor to create the deeptabular component
tab_preprocessor = TabPreprocessor(
    cat_embed_cols=cat_embed_cols,
    continuous_cols=continuous_cols
)

# For Training
X_tab = tab_preprocessor.fit_transform(train_df)  # x_train
target = train_df["resale_price"].values  # y_train

# Create the TabMlp model with 2 linear layers in the MLP, with 200 and 100 neurons respectively.
model = TabMlp(
    column_idx=tab_preprocessor.column_idx,
    cat_embed_input=tab_preprocessor.cat_embed_input,
    continuous_cols=continuous_cols,
    mlp_hidden_dims=[200, 100]
)

# Create a Trainer for the training of the created TabMlp model with the RMSE cost function
wide_deep_model = WideDeep(deeptabular=model)  # Combine the TabMlp model with any other models you want to use

Trainer_ = Trainer(
    wide_deep_model,
    objective="root_mean_squared_error",
    metrics=[R2Score],
    num_workers=0
)

# Train the model for 100 epochs using this trainer, keeping a batch size of 64.
Trainer_.fit(X_tab=X_tab, target=target, n_epochs=60, batch_size=64)


'''
# TODO: Enter your code here
# 4️⃣ Define features
target_col = "resale_price"

continuous_cols = [
    "dist_to_nearest_stn", "dist_to_dhoby", "degree_centrality",
    "eigenvector_centrality", "remaining_lease_years", "floor_area_sqm"
]

cat_embed_input = ["town", "full_address", "nearest_stn", "flat_model_type", "storey_range"]

# 5️⃣ Preprocess Data
preprocessor = TabPreprocessor(
    continuous_cols=continuous_cols,
    cat_embed_input=cat_embed_input
)

# ✅ Apply preprocessing BEFORE accessing `cat_embed_input`
X_train = preprocessor.fit_transform(train_df)
X_test = preprocessor.transform(test_df)

# ✅ Now it's safe to access `cat_embed_input`
cat_embed_input = preprocessor.cat_embed_input  # ✅ FIXED

y_train = train_df[target_col].values
y_test = test_df[target_col].values

# 6️⃣ Define the TabMlp model
deeptabular = TabMlp(
    column_idx=preprocessor.column_idx,
    cat_embed_input=cat_embed_input,  # ✅ FIXED: Now correctly defined
    continuous_cols=continuous_cols,
    mlp_hidden_dims=[50],  # 1 hidden layer with 50 neurons
    mlp_dropout=0.1
)

# 7️⃣ Create the WideDeep model
model = WideDeep(deeptabular=deeptabular)
# model = WideDeep(wide=wide, deeptabular=deeptabular)  # ✅ Includes both components

# 8️⃣ Set up the trainer
trainer = Trainer(model, objective="regression", metrics=[R2Score])

# 9️⃣ Train the model
trainer.fit(X_train, y_train, batch_size=1024, n_epochs=50)

# 🔟 Make predictions
y_pred = trainer.predict(X_test).flatten()
'''

/usr/local/lib/python3.11/dist-packages/pytorch_widedeep/preprocessing/tab_preprocessor.py:364: UserWarning: Continuous columns will not be normalised
  warnings.warn("Continuous columns will not be normalised")
epoch 60: 100%|██████████| 1366/1366 [00:12<00:00, 108.67it/s, loss=3.48e+4, metrics={'r2': 0.947}]


'\n# TODO: Enter your code here\n# 4️⃣ Define features\ntarget_col = "resale_price"\n\ncontinuous_cols = [\n    "dist_to_nearest_stn", "dist_to_dhoby", "degree_centrality",\n    "eigenvector_centrality", "remaining_lease_years", "floor_area_sqm"\n]\n\ncat_embed_input = ["town", "full_address", "nearest_stn", "flat_model_type", "storey_range"]\n\n# 5️⃣ Preprocess Data\npreprocessor = TabPreprocessor(\n    continuous_cols=continuous_cols,\n    cat_embed_input=cat_embed_input\n)\n\n# ✅ Apply preprocessing BEFORE accessing `cat_embed_input`\nX_train = preprocessor.fit_transform(train_df)\nX_test = preprocessor.transform(test_df)\n\n# ✅ Now it\'s safe to access `cat_embed_input`\ncat_embed_input = preprocessor.cat_embed_input  # ✅ FIXED\n\ny_train = train_df[target_col].values\ny_test = test_df[target_col].values\n\n# 6️⃣ Define the TabMlp model\ndeeptabular = TabMlp(\n    column_idx=preprocessor.column_idx,\n    cat_embed_input=cat_embed_input,  # ✅ FIXED: Now correctly defined\n    contin

>Report the test RMSE and the test R2 value that you obtained.

In [19]:
# TODO: Enter your code here
# 1️⃣1️⃣ Compute RMSE and R²
import math
from sklearn.metrics import r2_score, mean_squared_error

'''
from pytorch_widedeep.preprocessing import TabPreprocessor
from pytorch_widedeep.models import TabMlp, WideDeep
from pytorch_widedeep.metrics import R2Score
from pytorch_widedeep.training import Trainer  # ✅ Correct

# Define categorical and continuous columns
cat_embed_cols = ["town", "full_address", "nearest_stn", "flat_model_type", "storey_range"]
continuous_cols = ["dist_to_nearest_stn", "dist_to_dhoby", "degree_centrality", "eigenvector_centrality", "remaining_lease_years", "floor_area_sqm"]

# Use TabPreprocessor to create the deeptabular component
tab_preprocessor = TabPreprocessor(
    cat_embed_cols=cat_embed_cols,
    continuous_cols=continuous_cols
)
'''

x_test = tab_preprocessor.transform(test_df)
y_test = test_df['resale_price'].values

predictions = Trainer_.predict(X_tab=x_test,batch_size=64)

rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = R2Score()(torch.tensor(predictions), torch.tensor(y_test)).item()

print(f"Test RMSE: {rmse:.4f}")
print(f"Test R²: {r2:.4f}")

predict: 100%|██████████| 1128/1128 [00:04<00:00, 234.33it/s]


Test RMSE: 102343.6871
Test R²: 0.6341


Part B, Q3 (10 marks)
---
Besides ensuring that your neural network performs well, it is important to be able to explain the model’s decision. **Captum** is a very handy library that helps you to do so for PyTorch models.

Many model explainability algorithms for deep learning models are available in Captum. These algorithms are often used to generate an attribution score for each feature. Features with larger scores are more ‘important’ and some algorithms also provide information about directionality (i.e. a feature with very negative attribution scores means the larger the value of that feature, the lower the value of the output).

In general, these algorithms can be grouped into two paradigms:
- **perturbation based approaches** (e.g. Feature Ablation)
- **gradient / backpropagation based approaches** (e.g. Saliency)

The former adopts a brute-force approach of removing / permuting features one by one and does not scale up well. The latter depends on gradients and they can be computed relatively quickly. But unlike how backpropagation computes gradients with respect to weights, gradients here are computed **with respect to the input**. This gives us a sense of how much a change in the input affects the model’s outputs.




---



In [20]:
!pip install captum

In [21]:
from captum.attr import Saliency, InputXGradient, IntegratedGradients, GradientShap, FeatureAblation

<frozen importlib._bootstrap>:1047: ImportWarning: _PyDrive2ImportHook.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:1047: ImportWarning: _PyDriveImportHook.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:1047: ImportWarning: _GenerativeAIImportHook.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:1047: ImportWarning: _OpenCVImportHook.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:1047: ImportWarning: _BokehImportHook.find_spec() not found; falling back to find_module()


> First, use the train set (year 2020 and before) and test set (year 2021) following the splits in Question B1 (validation set is not required here). To keep things simple, we will **limit our analysis to numeric / continuous features only**. Drop all categorical features from the dataframes. Standardise the features via **StandardScaler** (fit to training set, then transform all).

In [ ]:
# TODO: Enter your code here

> Follow this tutorial to generate the plot from various model explainability algorithms (https://captum.ai/tutorials/House_Prices_Regression_Interpret).
Specifically, make the following changes:
- Use a feedforward neural network with 3 hidden layers, each having 5 neurons. Train using Adam optimiser with learning rate of 0.001.
- Use Input x Gradients, Integrated Gradients, DeepLift, GradientSHAP, Feature Ablation. To avoid long running time, you can limit the analysis to the first 1000 samples in test set.

In [ ]:
# TODO: Enter your code here

> Read the following [descriptions](https://captum.ai/docs/attribution_algorithms) and [comparisons](https://captum.ai/docs/algorithms_comparison_matrix) in Captum to build up your understanding of the difference of various explainability algorithms. Based on your plot, identify the three most important features for regression. Explain how each of these features influences the regression outcome.


\# TODO: \<Enter your answer here\>

Part B, Q4 (10 marks)
---

Model degradation is a common issue faced when deploying machine learning models (including neural networks) in the real world. New data points could exhibit a different pattern from older data points due to factors such as changes in government policy or market sentiments. For instance, housing prices in Singapore have been increasing and the Singapore government has introduced 3 rounds of cooling measures over the past years (16 December 2021, 30 September 2022, 27 April 2023).

In such situations, the distribution of the new data points could differ from the original data distribution which the models were trained on. Recall that machine learning models often work with the assumption that the test distribution should be similar to train distribution. When this assumption is violated, model performance will be adversely impacted.  In the last part of this assignment, we will investigate to what extent model degradation has occurred.




---



In [ ]:
!pip install alibi-detect

In [ ]:
from alibi_detect.cd import TabularDrift

> Evaluate your model from B1 on data from year 2022 and report the test R2.

In [ ]:
# TODO: Enter your code here

> Evaluate your model from B1 on data from year 2023 and report the test R2.

In [ ]:
# TODO: Enter your code here

> Did model degradation occur for the deep learning model?

\# TODO: \<Enter your answer here\>

Model degradation could be caused by [various data distribution shifts](https://huyenchip.com/2022/02/07/data-distribution-shifts-and-monitoring.html#data-shift-types): covariate shift (features), label shift and/or concept drift (altered relationship between features and labels).
There are various conflicting terminologies in the [literature](https://www.sciencedirect.com/science/article/pii/S0950705122002854#tbl1). Let’s stick to this reference for this assignment.

> Using the **Alibi Detect** library, apply the **TabularDrift** function with the training data (year 2020 and before) used as the reference and **detect which features have drifted** in the 2023 test dataset. Before running the statistical tests, ensure you **sample 1000 data points** each from the train and test data. Do not use the whole train/test data. (Hint: use this example as a guide https://docs.seldon.io/projects/alibi-detect/en/stable/examples/cd_chi2ks_adult.html)


In [ ]:
# TODO: Enter your code here

> Assuming that the flurry of housing measures have made an impact on the relationship between all the features and resale_price (i.e. P(Y|X) changes), which type of data distribution shift possibly led to model degradation?

\# TODO: \<Enter your answer here\>

> From your analysis via TabularDrift, which features contribute to this shift?

\# TODO: \<Enter your answer here\>

> Suggest 1 way to address model degradation and implement it, showing improved test R2 for year 2023.

\# TODO: \<Enter your answer here\>

In [ ]:
# TODO: Enter your code here